# MAXIM Trainer (No MoE)

Notebook para entrenar MAXIM base en dos modos:
- `single_task`: entrenamiento especializado (init aleatoria)
- `multi_task`: entrenamiento conjunto de todas las tareas (init desde checkpoint manual opcional; aleatoria si MULTITASK_INIT_CKPT="")

In [ ]:
%cd /content
!git clone https://github.com/Matiata/maxim.git
%cd /content/maxim
!pip install -r requirements.txt
!pip install -e .

/content
Cloning into 'maxim'...
remote: Enumerating objects: 385, done.
remote: Counting objects: 100% (285/285), done.
remote: Compressing objects: 100% (178/178), done.
remote: Total 385 (delta 186), reused 197 (delta 106), pack-reused 100 (from 1)
Receiving objects: 100% (385/385), 18.07 MiB | 15.29 MiB/s, done.
Resolving deltas: 100% (214/214), done.
/content/maxim
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.7/88.7 kB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 102.3/102.3 kB 9.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.8/135.8 kB 14.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 101.8/101.8 kB 12.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.1/58.1 kB 4.8 MB/s eta 0:00:00
  Attempting uninstall: toolz
    Found existing installation: toolz 0.12.1
    Uninstalling toolz-0.12.1:
      Successfully uninstalled toolz-0.12.1
  Attempting uninstall: absl-py
    Found existing installation: absl-py 1.4.0

In [ ]:
from google.colab import drive # works only for colab
drive.mount('/content/gdrive/',)

Mounted at /content/gdrive/


In [ ]:
import collections
import functools
import importlib
import io
import os
import re
import time
from typing import Any

import jax
import jax.numpy as jnp
from jax import random
import ml_collections
import numpy as np
import optax
from PIL import Image
import tensorflow as tf
from flax import traverse_util
from flax.core import freeze, unfreeze
from flax.training import train_state, checkpoints

# Runtime configuration

In [ ]:
TRAIN_MODE = "multi_task"  # "single_task" or "multi_task"
TASK = "enhance"            # Used only when TRAIN_MODE == "single_task"

TASKS = ["deblur", "dehaze", "denoise", "derain", "enhance"]

DATA_ROOT_DIR = "/content/gdrive/MyDrive/Facultad/tesis/Datasets/Classifier"
OUTPUT_ROOT_DIR = "/content/gdrive/MyDrive/Facultad/tesis/ckpts/maxim_no_moe"

BATCH_SIZE = 2
EVAL_BATCH_SIZE = 1
NUM_EPOCHS = 30
LEARNING_RATE = 2e-4
WARMUP_EPOCHS = 3
PATCH_SIZE = 256
# Eval en resolución completa puede provocar OOM con MAXIM.
# Usar PATCH_SIZE para validar por parches (recomendado) o None para full-res.
EVAL_PATCH_SIZE = PATCH_SIZE
LOG_EVERY = 10
SAVE_EVERY = 2
WEIGHT_DECAY = 1e-4
SEED = 42

SAMPLING_MODE_TRAIN = "uniform"
SAMPLING_MODE_VAL = "proportional"
EPOCH_DEFINITION = "proportional"
STEPS_PER_EPOCH_OVERRIDE = 2000  # Set to None to infer from dataset sizes.

# MULTITASK_INIT_CKPT = "/content/gdrive/MyDrive/Facultad/tesis/ckpts/ckpt_Enhancement_LOL.npz"  # warm-start (corrida original)
MULTITASK_INIT_CKPT = ""  # from-scratch multi_task (random init) -- comparacion justa vs MoE
MULTITASK_VARIANT = "S-2"  # escala del backbone multi-task: S-2 = misma que MoE (comparacion justa); usar S-3 para la corrida original
# RESUME_CHECKPOINT_PATH = ""  # Optional; relative paths are resolved inside OUTPUT_DIR.
RESUME_CHECKPOINT_PATH = "/content/gdrive/MyDrive/Facultad/tesis/ckpts/maxim_no_moe/multi_task_all_S-2_scratch/best_checkpoint/checkpoint_17"  # Optional; relative paths are resolved inside OUTPUT_DIR.

RUN_SANITY_CHECK = False
SANITY_STEPS = 2
START_TRAINING = True

_MODEL_CONFIGS = {
    "variant": "",
    "dropout_rate": 0.1,
    "num_outputs": 3,
    "use_bias": True,
    "num_supervision_scales": 3,
}
_MODEL_VARIANT_DICT = {
    "denoise": "S-3",
    "deblur": "S-3",
    "derain": "S-2",
    "dehaze": "S-2",
    "enhance": "S-2",
}

TASK_DIR_MAP = {task: os.path.join(DATA_ROOT_DIR, task) for task in TASKS}

if TRAIN_MODE not in {"single_task", "multi_task"}:
    raise ValueError(f"Unknown TRAIN_MODE: {TRAIN_MODE}")
if TASK not in TASK_DIR_MAP:
    raise ValueError(f"Unknown TASK: {TASK}")

if TRAIN_MODE == "single_task":
    ACTIVE_VARIANT = _MODEL_VARIANT_DICT[TASK]
    ACTIVE_TASK_DIRS = [TASK_DIR_MAP[TASK]]
    RUN_NAME = f"single_task_{TASK}"
else:
    ACTIVE_VARIANT = MULTITASK_VARIANT
    ACTIVE_TASK_DIRS = [TASK_DIR_MAP[t] for t in TASKS]
    init_tag = "scratch" if not MULTITASK_INIT_CKPT else "warm"
    RUN_NAME = f"multi_task_all_{ACTIVE_VARIANT}_{init_tag}"

OUTPUT_DIR = os.path.join(OUTPUT_ROOT_DIR, RUN_NAME)
BEST_OUTPUT_DIR = os.path.join(OUTPUT_DIR, "best_checkpoint")
if RESUME_CHECKPOINT_PATH and not os.path.isabs(RESUME_CHECKPOINT_PATH):
    RESUME_CHECKPOINT_PATH = os.path.join(OUTPUT_DIR, RESUME_CHECKPOINT_PATH)
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(BEST_OUTPUT_DIR, exist_ok=True)

print("=== Configuration ===")
print(f"TRAIN_MODE: {TRAIN_MODE}")
print(f"TASK: {TASK}")
print(f"ACTIVE_VARIANT: {ACTIVE_VARIANT}")
print(f"TASK_DIRS: {ACTIVE_TASK_DIRS}")
print(f"OUTPUT_DIR: {OUTPUT_DIR}")
print(f"MULTITASK_INIT_CKPT: {MULTITASK_INIT_CKPT}")
print(f"RESUME_CHECKPOINT_PATH: {RESUME_CHECKPOINT_PATH}")

=== Configuration ===
TRAIN_MODE: multi_task
TASK: enhance
ACTIVE_VARIANT: S-2
TASK_DIRS: ['/content/gdrive/MyDrive/Facultad/tesis/Datasets/Classifier/deblur', '/content/gdrive/MyDrive/Facultad/tesis/Datasets/Classifier/dehaze', '/content/gdrive/MyDrive/Facultad/tesis/Datasets/Classifier/denoise', '/content/gdrive/MyDrive/Facultad/tesis/Datasets/Classifier/derain', '/content/gdrive/MyDrive/Facultad/tesis/Datasets/Classifier/enhance']
OUTPUT_DIR: /content/gdrive/MyDrive/Facultad/tesis/ckpts/maxim_no_moe/multi_task_all_S-2_scratch
MULTITASK_INIT_CKPT: 
RESUME_CHECKPOINT_PATH: /content/gdrive/MyDrive/Facultad/tesis/ckpts/maxim_no_moe/multi_task_all_S-2_scratch/best_checkpoint/checkpoint_17


# Auxiliar Functions


In [ ]:
def recover_tree(keys, values):
    """Recover nested dict from flat checkpoint keys."""
    tree = {}
    sub_trees = collections.defaultdict(list)
    for k, v in zip(keys, values):
        if "/" not in k:
            tree[k] = v
        else:
            k_left, k_right = k.split("/", 1)
            sub_trees[k_left].append((k_right, v))
    for k, kv_pairs in sub_trees.items():
        k_subtree, v_subtree = zip(*kv_pairs)
        tree[k] = recover_tree(k_subtree, v_subtree)
    return tree


def to_frozen_params(params):
    """Normalize params to FrozenDict to keep JAX treedef stable."""
    return freeze(unfreeze(params))


def reset_optimizer_with_params(state, params):
    """Rebuild optimizer state so it matches loaded param structure."""
    params = to_frozen_params(params)
    step_dtype = jnp.asarray(state.step).dtype
    step0 = jnp.asarray(0, dtype=step_dtype)
    return state.replace(params=params, opt_state=state.tx.init(params), step=step0)


def get_npz_params(ckpt_path):
    """Load MAXIM params from original .npz checkpoint format."""
    with tf.io.gfile.GFile(ckpt_path, "rb") as f:
        data = f.read()
    values = np.load(io.BytesIO(data), allow_pickle=False)
    params = recover_tree(*zip(*values.items()))
    if "opt" in params and "target" in params["opt"]:
        return params["opt"]["target"]
    return params


def load_matching_params(target_params, source_params):
    """Load only matching params and report coverage."""
    flat_target = traverse_util.flatten_dict(unfreeze(target_params))
    flat_source = traverse_util.flatten_dict(source_params)

    updated = dict(flat_target)
    loaded = 0
    skipped_shape = 0

    for key, value in flat_source.items():
        if key in flat_target:
            if flat_target[key].shape == value.shape:
                updated[key] = value
                loaded += 1
            else:
                skipped_shape += 1

    merged = freeze(traverse_util.unflatten_dict(updated))
    return merged, {"loaded": loaded, "skipped_shape": skipped_shape, "total_target": len(flat_target)}


def initialize_state_by_mode(state):
    """Apply requested init policy:
    - single_task: keep random init
    - multi_task: load manual checkpoint path, or random init if MULTITASK_INIT_CKPT is empty
    """
    if TRAIN_MODE == "single_task":
        print("single_task mode: using random initialization.")
        return reset_optimizer_with_params(state, state.params)

    if not MULTITASK_INIT_CKPT:
        print("multi_task mode: MULTITASK_INIT_CKPT empty -> using random initialization.")
        return reset_optimizer_with_params(state, state.params)

    ckpt_path = MULTITASK_INIT_CKPT
    if ckpt_path.endswith(".npz"):
        print(f"Loading multi-task init from NPZ: {ckpt_path}")
        source_params = get_npz_params(ckpt_path)
        new_params, report = load_matching_params(state.params, source_params)
        if report["loaded"] == 0:
            raise ValueError(
                "No compatible parameters were loaded from NPZ checkpoint. "
                "Check variant and checkpoint path."
            )
        print(
            f"Loaded {report['loaded']} / {report['total_target']} params "
            f"(shape-skipped: {report['skipped_shape']})."
        )
        return reset_optimizer_with_params(state, new_params)

    print(f"Loading multi-task init from Flax checkpoint dir/file: {ckpt_path}")
    try:
        restored = checkpoints.restore_checkpoint(ckpt_dir=ckpt_path, target=state)
    except Exception as err:
        raise ValueError(
            f"Failed to restore multi-task checkpoint from '{ckpt_path}': {err}"
        ) from err

    if restored is None:
        raise ValueError(f"Checkpoint path '{ckpt_path}' did not return a valid state.")
    if not hasattr(restored, "params"):
        raise ValueError(
            "Restored object does not expose 'params'. "
            f"Got type: {type(restored)}"
        )

    print("Multi-task checkpoint restored successfully.")
    return reset_optimizer_with_params(restored, restored.params)

def checkpoint_path_exists(path):
    """Return whether a local/GCS/Drive checkpoint path exists."""
    return bool(path) and tf.io.gfile.exists(path)


def restore_training_state_or_fail(state, ckpt_path):
    """Restore a full TrainState and fail loudly if nothing was loaded."""
    if not ckpt_path:
        return state
    if not checkpoint_path_exists(ckpt_path):
        raise FileNotFoundError(f"Resume checkpoint path does not exist: {ckpt_path}")

    before_step = int(jax.device_get(state.step))
    try:
        restored = checkpoints.restore_checkpoint(ckpt_dir=ckpt_path, target=state)
    except Exception as err:
        raise ValueError(f"Failed to restore resume checkpoint from {ckpt_path}: {err}") from err

    if restored is None or not hasattr(restored, "params"):
        raise ValueError(f"Resume checkpoint did not return a valid TrainState: {ckpt_path}")

    restored_step = int(jax.device_get(restored.step))
    if restored_step == before_step:
        raise ValueError(
            "Resume checkpoint restore left state.step unchanged. "
            "This usually means the path is a directory without a readable checkpoint. "
            f"Path: {ckpt_path}"
        )

    print(f"Resumed full state from: {ckpt_path}")
    print(f"Restored optimizer step: {restored_step}")
    return restored


def infer_start_epoch_from_resume(ckpt_path, state, steps_per_epoch):
    """Infer next epoch from checkpoint filename, falling back to optimizer step."""
    match = re.search(r"checkpoint_(\d+)$", ckpt_path.rstrip("/"))
    if match:
        return int(match.group(1))

    latest = checkpoints.latest_checkpoint(ckpt_path)
    if latest:
        match = re.search(r"checkpoint_(\d+)$", latest.rstrip("/"))
        if match:
            return int(match.group(1))

    optimizer_steps = int(jax.device_get(state.step))
    if optimizer_steps > 0:
        return optimizer_steps // steps_per_epoch

    raise ValueError(f"Could not infer start epoch from resume checkpoint: {ckpt_path}")


In [ ]:
def load_image(filepath, max_retries=5, retry_delay=0.2):
    """Load image robustly, retrying transient I/O failures."""
    last_error = None
    for attempt in range(max_retries):
        try:
            with Image.open(filepath) as img:
                rgb = img.convert("RGB")
                return np.asarray(rgb, np.float32) / 255.0
        except OSError as err:
            last_error = err
            # Retry transient storage errors (e.g. network-mounted drives).
            if attempt < max_retries - 1:
                time.sleep(retry_delay)
                continue
            raise OSError(
                f"Failed to read image after {max_retries} attempts: {filepath}"
            ) from err

    raise OSError(f"Unexpected image loading failure: {filepath}") from last_error


def make_shape_even(image):
    h, w = image.shape[:2]
    padh = 1 if h % 2 != 0 else 0
    padw = 1 if w % 2 != 0 else 0
    if padh or padw:
        image = np.pad(image, [(0, padh), (0, padw), (0, 0)], mode="reflect")
    return image


def mod_padding_symmetric(image, factor=64):
    h, w = image.shape[:2]
    h_pad = ((h + factor - 1) // factor) * factor
    w_pad = ((w + factor - 1) // factor) * factor
    padh = h_pad - h
    padw = w_pad - w
    if padh or padw:
        image = np.pad(
            image,
            [(padh // 2, padh - padh // 2), (padw // 2, padw - padw // 2), (0, 0)],
            mode="reflect",
        )
    return image


def resize_to_match(image, target):
    """Pad/crop image to target shape if they differ."""
    h, w = image.shape[:2]
    th, tw = target.shape[:2]

    if h == tw and w == th:
        image = np.rot90(image)
        h, w = image.shape[:2]

    pad_h = max(0, th - h)
    pad_w = max(0, tw - w)
    if pad_h or pad_w:
        image = np.pad(image, ((0, pad_h), (0, pad_w), (0, 0)), mode="reflect")
        h, w = image.shape[:2]

    if h > th or w > tw:
        top = max(0, (h - th) // 2)
        left = max(0, (w - tw) // 2)
        image = image[top:top + th, left:left + tw]

    return image


def random_crop(image, target, crop_size):
    h_i, w_i = image.shape[:2]
    h_t, w_t = target.shape[:2]

    if h_i < crop_size or w_i < crop_size:
        image = np.pad(
            image,
            ((0, max(0, crop_size - h_i)), (0, max(0, crop_size - w_i)), (0, 0)),
            mode="reflect",
        )
    if h_t < crop_size or w_t < crop_size:
        target = np.pad(
            target,
            ((0, max(0, crop_size - h_t)), (0, max(0, crop_size - w_t)), (0, 0)),
            mode="reflect",
        )

    h, w = image.shape[:2]
    top = np.random.randint(0, h - crop_size + 1)
    left = np.random.randint(0, w - crop_size + 1)

    image = image[top:top + crop_size, left:left + crop_size]
    target = target[top:top + crop_size, left:left + crop_size]
    return image, target


def center_crop(image, target, crop_size):
    h_i, w_i = image.shape[:2]
    h_t, w_t = target.shape[:2]

    if h_i < crop_size or w_i < crop_size:
        image = np.pad(
            image,
            ((0, max(0, crop_size - h_i)), (0, max(0, crop_size - w_i)), (0, 0)),
            mode="reflect",
        )
    if h_t < crop_size or w_t < crop_size:
        target = np.pad(
            target,
            ((0, max(0, crop_size - h_t)), (0, max(0, crop_size - w_t)), (0, 0)),
            mode="reflect",
        )

    h, w = image.shape[:2]
    top = max(0, (h - crop_size) // 2)
    left = max(0, (w - crop_size) // 2)

    image = image[top:top + crop_size, left:left + crop_size]
    target = target[top:top + crop_size, left:left + crop_size]
    return image, target


def random_flip(image, target):
    if np.random.rand() > 0.5:
        image = np.fliplr(image)
        target = np.fliplr(target)
    if np.random.rand() > 0.5:
        image = np.flipud(image)
        target = np.flipud(target)
    return image, target


def random_rotation(image, target):
    k = np.random.randint(0, 4)
    return np.rot90(image, k=k), np.rot90(target, k=k)


def read_lines_from_file(basepath, filepath):
    with open(filepath, "r", encoding="utf-8") as f:
        lines = [line.strip() for line in f if line.strip()]

    existing = []
    missing = []
    for line in lines:
        full = os.path.join(basepath, line)
        if os.path.exists(full):
            existing.append(full)
        else:
            missing.append(full)

    print(
        f"Checked {len(lines)} files in {basepath}: "
        f"{len(existing)} existing, {len(missing)} missing."
    )
    if missing:
        print(f"Missing files sample ({min(5, len(missing))}): {missing[:5]}")

    return existing, missing


def create_dataset(data_dir, batch_size, patch_size, is_training=True):
    print(
        f"Creating {'training' if is_training else 'validation'} dataset from {data_dir}"
    )
    input_dir = os.path.join(data_dir, "imgs")
    target_dir = os.path.join(data_dir, "GT")
    files_list = os.path.join(data_dir, "train.txt" if is_training else "test.txt")

    input_files, missing_inputs = read_lines_from_file(input_dir, files_list)
    target_files, missing_targets = read_lines_from_file(target_dir, files_list)
    if missing_inputs or missing_targets:
        raise FileNotFoundError("Some dataset files listed in txt are missing.")

    def load_and_preprocess(input_path, target_path):
        inp = load_image(input_path.numpy().decode())
        tgt = load_image(target_path.numpy().decode())

        inp = resize_to_match(inp, tgt)
        tgt = resize_to_match(tgt, inp)

        orig_h, orig_w = inp.shape[:2]

        inp = make_shape_even(inp)
        tgt = make_shape_even(tgt)
        even_h, even_w = inp.shape[:2]

        inp = mod_padding_symmetric(inp, factor=64)
        tgt = mod_padding_symmetric(tgt, factor=64)
        pad_h, pad_w = inp.shape[:2]

        if is_training:
            inp, tgt = random_crop(inp, tgt, patch_size)
            inp, tgt = random_flip(inp, tgt)
            inp, tgt = random_rotation(inp, tgt)
        elif patch_size is not None:
            # Evita OOM en evaluación usando parches de tamaño fijo.
            inp, tgt = center_crop(inp, tgt, patch_size)

        sizes = np.array([orig_h, orig_w, even_h, even_w, pad_h, pad_w], dtype=np.int32)
        return inp.astype(np.float32), tgt.astype(np.float32), sizes

    dataset = tf.data.Dataset.from_tensor_slices((input_files, target_files))
    if is_training:
        dataset = dataset.shuffle(buffer_size=1000)

    parallel_reads = min(4, os.cpu_count() or 1)
    dataset = dataset.map(
        lambda x, y: tf.py_function(
            func=load_and_preprocess,
            inp=[x, y],
            Tout=[tf.float32, tf.float32, tf.int32],
        ),
        num_parallel_calls=parallel_reads,
    )

    def set_shapes(inp, tgt, sizes):
        inp.set_shape([None, None, 3])
        tgt.set_shape([None, None, 3])
        sizes.set_shape([6])
        return inp, tgt, sizes

    dataset = dataset.map(set_shapes, num_parallel_calls=tf.data.AUTOTUNE)
    dataset = dataset.batch(batch_size, drop_remainder=is_training)
    dataset = dataset.prefetch(tf.data.AUTOTUNE)
    return dataset, len(input_files)


def create_dataset_unbatched(data_dir, patch_size, is_training=True):
    ds, length = create_dataset(
        data_dir=data_dir,
        batch_size=1,
        patch_size=patch_size,
        is_training=is_training,
    )
    return ds.unbatch(), length


def create_unified_dataset(
    task_dirs,
    batch_size,
    patch_size,
    is_training=True,
    sampling_mode="uniform",
    shuffle_buffer=1000,
    epoch_definition="proportional",
):
    datasets = []
    lengths = []

    for data_dir in task_dirs:
        ds, n = create_dataset_unbatched(
            data_dir=data_dir,
            patch_size=patch_size,
            is_training=is_training,
        )
        datasets.append(ds)
        lengths.append(n)

    lengths = tf.constant(lengths, dtype=tf.float32)

    if sampling_mode == "uniform":
        weights = tf.ones_like(lengths) / tf.cast(tf.size(lengths), tf.float32)
    elif sampling_mode == "proportional":
        weights = lengths / tf.reduce_sum(lengths)
    else:
        raise ValueError(f"Unknown sampling_mode: {sampling_mode}")

    unified_ds = tf.data.Dataset.sample_from_datasets(
        datasets,
        weights=weights,
        stop_on_empty_dataset=False,
    )

    if is_training:
        unified_ds = unified_ds.shuffle(shuffle_buffer).repeat()

    unified_ds = unified_ds.batch(batch_size, drop_remainder=is_training)
    unified_ds = unified_ds.prefetch(tf.data.AUTOTUNE)

    if STEPS_PER_EPOCH_OVERRIDE is not None:
        steps_per_epoch = int(STEPS_PER_EPOCH_OVERRIDE)
    else:
        if epoch_definition == "proportional":
            epoch_samples = tf.reduce_sum(lengths)
        elif epoch_definition == "balanced":
            epoch_samples = tf.reduce_max(lengths) * len(task_dirs)
        else:
            raise ValueError(f"Unknown epoch_definition: {epoch_definition}")
        steps_per_epoch = max(1, int(epoch_samples.numpy() // batch_size))

    return unified_ds, steps_per_epoch

In [ ]:
class TrainState(train_state.TrainState):
    """Extended train state with optional batch statistics."""
    batch_stats: Any = None


def resize_target_to(pred, target):
    if pred.shape == target.shape:
        return target
    scale_h = target.shape[1] // pred.shape[1]
    scale_w = target.shape[2] // pred.shape[2]
    return target[:, ::scale_h, ::scale_w, :]


def compute_loss(preds, targets, num_scales=3):
    total_loss = 0.0

    if isinstance(preds, list):
        for stage_preds in preds:
            if isinstance(stage_preds, list):
                for scale_idx, pred in enumerate(stage_preds):
                    weight = 0.5 ** (num_scales - scale_idx - 1)
                    tgt_resized = resize_target_to(pred, targets)
                    total_loss += weight * jnp.mean(jnp.abs(pred - tgt_resized))
            else:
                tgt_resized = resize_target_to(stage_preds, targets)
                total_loss += jnp.mean(jnp.abs(stage_preds - tgt_resized))
    else:
        tgt_resized = resize_target_to(preds, targets)
        total_loss = jnp.mean(jnp.abs(preds - tgt_resized))

    return total_loss


def final_prediction(preds):
    if isinstance(preds, list):
        stage = preds[-1]
        if isinstance(stage, list):
            return stage[-1]
        return stage
    return preds


def compute_psnr(pred, target):
    pred_255 = pred * 255.0
    target_255 = target * 255.0
    mse = jnp.mean((pred_255 - target_255) ** 2)
    mse = jnp.maximum(mse, 1e-6)
    return 20.0 * jnp.log10(255.0 / jnp.sqrt(mse))


def create_learning_rate_schedule(base_lr, warmup_epochs, total_steps, steps_per_epoch):
    warmup_steps = int(warmup_epochs * steps_per_epoch)
    warmup_fn = optax.linear_schedule(
        init_value=0.0,
        end_value=base_lr,
        transition_steps=max(1, warmup_steps),
    )
    cosine_fn = optax.cosine_decay_schedule(
        init_value=base_lr,
        decay_steps=max(1, total_steps - warmup_steps),
        alpha=1e-6,
    )
    return optax.join_schedules([warmup_fn, cosine_fn], boundaries=[warmup_steps])


def build_model(variant):
    maxim_mod = importlib.import_module("maxim.models.maxim")
    maxim_configs = ml_collections.ConfigDict(_MODEL_CONFIGS)
    maxim_configs.variant = variant
    return maxim_mod.Model(**maxim_configs)


def create_train_state(rng, model, learning_rate_fn, weight_decay):
    dummy_input = jnp.ones([1, PATCH_SIZE, PATCH_SIZE, 3])
    variables = model.init(rng, dummy_input, train=True)
    params = variables["params"]
    batch_stats = variables.get("batch_stats", None)

    tx = optax.chain(
        optax.clip_by_global_norm(1.0),
        optax.adamw(learning_rate=learning_rate_fn, weight_decay=weight_decay),
    )

    return TrainState.create(
        apply_fn=model.apply,
        params=params,
        tx=tx,
        batch_stats=batch_stats,
    )


@functools.partial(jax.jit, static_argnums=(3,))
def train_step(state, batch_input, batch_target, num_scales, rng):
    def loss_fn(params):
        rngs = {"dropout": rng}
        if state.batch_stats is not None:
            preds, updates = state.apply_fn(
                {"params": params, "batch_stats": state.batch_stats},
                batch_input,
                train=True,
                rngs=rngs,
                mutable=["batch_stats"],
            )
            new_batch_stats = updates["batch_stats"]
        else:
            preds = state.apply_fn({"params": params}, batch_input, train=True, rngs=rngs)
            new_batch_stats = None

        loss = compute_loss(preds, batch_target, num_scales=num_scales)
        pred_final = final_prediction(preds)
        tgt_final = resize_target_to(pred_final, batch_target)
        psnr = compute_psnr(pred_final, tgt_final)

        return loss, (psnr, new_batch_stats)

    (loss, (psnr, new_batch_stats)), grads = jax.value_and_grad(loss_fn, has_aux=True)(state.params)
    state = state.apply_gradients(grads=grads)
    if new_batch_stats is not None:
        state = state.replace(batch_stats=new_batch_stats)

    metrics = {"loss": loss, "psnr": psnr}
    return state, metrics


@jax.jit
def eval_step(state, batch_input, batch_target):
    if state.batch_stats is not None:
        preds = state.apply_fn(
            {"params": state.params, "batch_stats": state.batch_stats},
            batch_input,
            train=False,
        )
    else:
        preds = state.apply_fn({"params": state.params}, batch_input, train=False)

    pred_final = final_prediction(preds)
    tgt_final = resize_target_to(pred_final, batch_target)
    loss = jnp.mean(jnp.abs(pred_final - tgt_final))
    psnr = compute_psnr(pred_final, tgt_final)
    return {"loss": loss, "psnr": psnr}


def train_epoch(state, train_iterator, num_scales, epoch, steps_per_epoch):
    batch_metrics = []
    print(f"Starting training epoch {epoch}")

    for step in range(steps_per_epoch):
        try:
            batch = next(train_iterator)
        except StopIteration:
            print("Dataset iterator exhausted; rebuilding iterator.")
            break

        batch_input, batch_target, _ = batch
        batch_input = jnp.array(batch_input)
        batch_target = jnp.array(batch_target)

        if batch_input.shape != batch_target.shape:
            print(f"Skipping step {step} due to shape mismatch: {batch_input.shape} vs {batch_target.shape}")
            continue

        rng = jax.random.fold_in(jax.random.PRNGKey(epoch), step)
        state, metrics = train_step(state, batch_input, batch_target, num_scales, rng)
        batch_metrics.append(metrics)

        if (step + 1) % LOG_EVERY == 0:
            m = jax.device_get(metrics)
            print(
                f"Epoch {epoch}, Step {step + 1}: "
                f"loss = {m['loss']:.4f}, psnr = {m['psnr']:.2f} dB"
            )

    if not batch_metrics:
        return state, {}

    epoch_metrics = {
        k: np.mean([jax.device_get(m[k]) for m in batch_metrics])
        for k in batch_metrics[0].keys()
    }
    return state, epoch_metrics


def evaluate(state, val_dataset, log_every=100):
    total_loss = 0.0
    total_psnr = 0.0
    n_batches = 0
    start = time.time()
    print("Starting evaluation...")

    for batch in val_dataset:
        batch_input, batch_target, _ = batch
        batch_input = jnp.array(batch_input)
        batch_target = jnp.array(batch_target)

        metrics = eval_step(state, batch_input, batch_target)
        metrics = jax.device_get(metrics)

        total_loss += float(metrics["loss"])
        total_psnr += float(metrics["psnr"])
        n_batches += 1

        if n_batches % log_every == 0:
            elapsed = time.time() - start
            rate = n_batches / elapsed if elapsed > 0 else 0.0
            print(
                f"  [eval] {n_batches} batches | "
                f"running psnr = {total_psnr / n_batches:.2f} dB, "
                f"loss = {total_loss / n_batches:.4f} | {rate:.1f} batch/s"
            )

    if n_batches == 0:
        print("Evaluation: no batches processed.")
        return {}

    elapsed = time.time() - start
    print(
        f"Evaluation done: {n_batches} batches in {elapsed:.1f}s | "
        f"psnr = {total_psnr / n_batches:.2f} dB, loss = {total_loss / n_batches:.4f}"
    )
    return {
        "loss": total_loss / n_batches,
        "psnr": total_psnr / n_batches,
    }

# Build datasets, model and state

In [ ]:
rng = random.PRNGKey(SEED)
num_scales = _MODEL_CONFIGS["num_supervision_scales"]

if TRAIN_MODE == "single_task":
    task_dir = ACTIVE_TASK_DIRS[0]

    train_dataset, train_size = create_dataset(
        data_dir=task_dir,
        batch_size=BATCH_SIZE,
        patch_size=PATCH_SIZE,
        is_training=True,
    )
    train_dataset = train_dataset.repeat()

    val_dataset, val_size = create_dataset(
        data_dir=task_dir,
        batch_size=EVAL_BATCH_SIZE,
        patch_size=EVAL_PATCH_SIZE,
        is_training=False,
    )

    inferred_steps = max(1, train_size // BATCH_SIZE)
    steps_per_epoch = int(STEPS_PER_EPOCH_OVERRIDE or inferred_steps)

else:
    # create_unified_dataset already uses STEPS_PER_EPOCH_OVERRIDE
    train_dataset, steps_per_epoch = create_unified_dataset(
        task_dirs=ACTIVE_TASK_DIRS,
        batch_size=BATCH_SIZE,
        patch_size=PATCH_SIZE,
        is_training=True,
        sampling_mode=SAMPLING_MODE_TRAIN,
        epoch_definition=EPOCH_DEFINITION,
    )

    val_dataset, _ = create_unified_dataset(
        task_dirs=ACTIVE_TASK_DIRS,
        batch_size=EVAL_BATCH_SIZE,
        patch_size=EVAL_PATCH_SIZE,
        is_training=False,
        sampling_mode=SAMPLING_MODE_VAL,
        epoch_definition=EPOCH_DEFINITION,
    )

print(f"Steps per epoch: {steps_per_epoch}")
total_steps = steps_per_epoch * NUM_EPOCHS

learning_rate_fn = create_learning_rate_schedule(
    base_lr=LEARNING_RATE,
    warmup_epochs=WARMUP_EPOCHS,
    total_steps=total_steps,
    steps_per_epoch=steps_per_epoch,
)

model = build_model(ACTIVE_VARIANT)
rng, init_rng = random.split(rng)
state = create_train_state(init_rng, model, learning_rate_fn, WEIGHT_DECAY)
start_epoch = 0

if RESUME_CHECKPOINT_PATH:
    state = restore_training_state_or_fail(state, RESUME_CHECKPOINT_PATH)
    start_epoch = infer_start_epoch_from_resume(RESUME_CHECKPOINT_PATH, state, steps_per_epoch)
    print(f"Continuing training from epoch {start_epoch + 1}")
else:
    state = initialize_state_by_mode(state)

train_dataset_iterator = iter(train_dataset)
print("Setup complete.")

Creating training dataset from /content/gdrive/MyDrive/Facultad/tesis/Datasets/Classifier/deblur
Checked 8680 files in /content/gdrive/MyDrive/Facultad/tesis/Datasets/Classifier/deblur/imgs: 8680 existing, 0 missing.
Checked 8680 files in /content/gdrive/MyDrive/Facultad/tesis/Datasets/Classifier/deblur/GT: 8680 existing, 0 missing.
Creating training dataset from /content/gdrive/MyDrive/Facultad/tesis/Datasets/Classifier/dehaze
Checked 909 files in /content/gdrive/MyDrive/Facultad/tesis/Datasets/Classifier/dehaze/imgs: 909 existing, 0 missing.
Checked 909 files in /content/gdrive/MyDrive/Facultad/tesis/Datasets/Classifier/dehaze/GT: 909 existing, 0 missing.
Creating training dataset from /content/gdrive/MyDrive/Facultad/tesis/Datasets/Classifier/denoise
Checked 142 files in /content/gdrive/MyDrive/Facultad/tesis/Datasets/Classifier/denoise/imgs: 142 existing, 0 missing.
Checked 142 files in /content/gdrive/MyDrive/Facultad/tesis/Datasets/Classifier/denoise/GT: 142 existing, 0 missing.


Resumed full state from: /content/gdrive/MyDrive/Facultad/tesis/ckpts/maxim_no_moe/multi_task_all_S-2_scratch/best_checkpoint/checkpoint_17
Restored optimizer step: 34000
Continuing training from epoch 18
Setup complete.


# Sanity check (1-2 train steps)

In [ ]:

if RUN_SANITY_CHECK:
    sanity_state = state
    sanity_iterator = iter(train_dataset)

    print("Running sanity check...")
    for sanity_step in range(SANITY_STEPS):
        batch_input, batch_target, _ = next(sanity_iterator)
        batch_input = jnp.array(batch_input)
        batch_target = jnp.array(batch_target)
        rng = jax.random.fold_in(jax.random.PRNGKey(SEED + 999), sanity_step)
        sanity_state, sanity_metrics = train_step(
            sanity_state,
            batch_input,
            batch_target,
            num_scales,
            rng,
        )
        sanity_metrics = jax.device_get(sanity_metrics)
        print(
            f"Sanity step {sanity_step + 1}/{SANITY_STEPS}: "
            f"loss={sanity_metrics['loss']:.4f}, "
            f"psnr={sanity_metrics['psnr']:.2f} dB"
        )

    print("Sanity check finished successfully.")
else:
    print("Sanity check skipped. Set RUN_SANITY_CHECK=True to run it.")

Sanity check skipped. Set RUN_SANITY_CHECK=True to run it.


# Full training loop

In [ ]:
if START_TRAINING:
    print("Starting training...")
    best_psnr = -np.inf

    train_dataset_iterator = iter(train_dataset)

    for epoch in range(start_epoch, NUM_EPOCHS):
        real_epoch = epoch + 1

        state, train_metrics = train_epoch(
            state=state,
            train_iterator=train_dataset_iterator,
            num_scales=num_scales,
            epoch=real_epoch,
            steps_per_epoch=steps_per_epoch,
        )
        print(f"Epoch {real_epoch} train metrics: {train_metrics}")

        val_metrics = evaluate(state, val_dataset)
        print(f"Epoch {real_epoch} val metrics: {val_metrics}")

        if real_epoch % SAVE_EVERY == 0:
            checkpoints.save_checkpoint(
                ckpt_dir=OUTPUT_DIR,
                target=state,
                step=real_epoch,
                overwrite=True,
                keep=5,
            )
            print(f"Saved checkpoint for epoch {real_epoch} in {OUTPUT_DIR}")

        current_psnr = val_metrics.get("psnr", -np.inf) if val_metrics else -np.inf
        if current_psnr > best_psnr:
            best_psnr = current_psnr
            checkpoints.save_checkpoint(
                ckpt_dir=BEST_OUTPUT_DIR,
                target=state,
                step=real_epoch,
                overwrite=True,
                keep=1,
            )
            print(f"Updated best checkpoint (PSNR={best_psnr:.2f}) at epoch {real_epoch}")

    print("Training finished.")
else:
    print("Training is disabled. Set START_TRAINING=True to run.")

Starting training...
Starting training epoch 18
Epoch 18, Step 10: loss = 0.1579, psnr = 24.27 dB
Epoch 18, Step 20: loss = 0.2513, psnr = 19.84 dB
Epoch 18, Step 30: loss = 0.1568, psnr = 24.96 dB
Epoch 18, Step 40: loss = 0.4021, psnr = 16.34 dB
Epoch 18, Step 50: loss = 0.2195, psnr = 21.65 dB
Epoch 18, Step 60: loss = 0.3123, psnr = 18.92 dB
Epoch 18, Step 70: loss = 0.1725, psnr = 23.56 dB
Epoch 18, Step 80: loss = 0.1330, psnr = 25.51 dB
Epoch 18, Step 90: loss = 0.1914, psnr = 24.08 dB
Epoch 18, Step 100: loss = 0.1628, psnr = 24.71 dB
Epoch 18, Step 110: loss = 0.1655, psnr = 23.47 dB
Epoch 18, Step 120: loss = 0.2798, psnr = 19.66 dB
Epoch 18, Step 130: loss = 0.1781, psnr = 24.16 dB
Epoch 18, Step 140: loss = 0.5155, psnr = 13.66 dB
Epoch 18, Step 150: loss = 0.1431, psnr = 24.22 dB
Epoch 18, Step 160: loss = 0.1447, psnr = 24.52 dB
Epoch 18, Step 170: loss = 0.4905, psnr = 15.17 dB
Epoch 18, Step 180: loss = 0.1097, psnr = 27.01 dB
Epoch 18, Step 190: loss = 0.2050, psnr = 2